# 🏦 Risk Lookup Function
### Climate Risk Profile for Banks & Insurers

This notebook provides a simple `get_risk_profile(city, year)` function that
returns the full multi-hazard climate risk profile for any city in the system.

**Use case:** A bank or insurer calls this function before approving a real estate
loan to assess the climate risk exposure of the property location.

---
## Step 1 — Load Composite Risk Data

In [ ]:
import pandas as pd
import os

OUTPUT_DIR = "../data/outputs"

# Load the composite risk scores table
composite = pd.read_csv(os.path.join(OUTPUT_DIR, "composite_risk_scores.csv"))

print(f"Loaded composite data: {len(composite)} rows, {composite['city'].nunique()} cities")
print(f"Available cities: {sorted(composite['city'].unique())}")
print(f"Available years:  {sorted(composite['year'].unique())}")

---
## Step 2 — Define the Risk Lookup Function

In [ ]:
def get_bank_recommendation(category: str) -> str:
    """
    Return a bank/insurer recommendation based on the composite risk category.
    """
    recommendations = {
        "LOW":       "LOW RISK — standard loan processing applicable",
        "MEDIUM":    "MEDIUM RISK — recommend annual climate review clause",
        "HIGH":      "HIGH RISK — recommend climate risk insurance clause before approving loan",
        "VERY HIGH": "VERY HIGH RISK — recommend independent climate assessment before approval",
    }
    return recommendations.get(category, "UNKNOWN — category not recognized")


def get_risk_profile(city: str, year: int) -> dict:
    """
    Given a city name and year, return the full multi-hazard risk profile.

    Parameters
    ----------
    city : str
        City name (case-insensitive, e.g. "Mumbai", "mumbai")
    year : int
        Year to look up (2015–2023)

    Returns
    -------
    dict with keys:
        city, year, drought_score, flood_score, heatwave_score,
        cyclone_score, landslide_score, composite_score,
        composite_category, bank_recommendation
    """
    # Case-insensitive city matching
    city_lower = city.strip().lower()
    match = composite[composite["city"].str.lower() == city_lower]

    if match.empty:
        available = sorted(composite["city"].unique())
        return {"error": f"City '{city}' not found. Available cities: {available}"}

    row = match[match["year"] == year]

    if row.empty:
        available_years = sorted(match["year"].unique())
        return {"error": f"Year {year} not available for {city}. Available years: {available_years}"}

    row = row.iloc[0]
    category = row["composite_category"]

    return {
        "city":                row["city"],
        "year":                int(row["year"]),
        "drought_score":       round(row.get("drought_risk_score", 0), 2),
        "flood_score":         round(row.get("flood_risk_score", 0), 2),
        "heatwave_score":      round(row.get("heatwave_risk_score", 0), 2),
        "cyclone_score":       round(row.get("cyclone_risk_score", 0), 2),
        "landslide_score":     round(row.get("landslide_risk_score", 0), 2),
        "composite_score":     round(row["composite_score"], 2),
        "composite_category":  category,
        "bank_recommendation": get_bank_recommendation(category),
    }


print("✓ get_risk_profile() function defined")

---
## Step 3 — Example Lookups

In [ ]:
import json

# ----- Demo lookups -----
demo_queries = [
    ("Mumbai", 2023),
    ("Chennai", 2022),
    ("Jodhpur", 2019),
    ("Shimla", 2023),
    ("Bengaluru", 2023),
]

for city, year in demo_queries:
    profile = get_risk_profile(city, year)
    print(f"\n{'='*60}")
    print(f"  {city} ({year})")
    print(f"{'='*60}")
    print(json.dumps(profile, indent=2))

---
## Step 4 — Batch Lookup: All Cities for Latest Year

In [ ]:
# ----- Look up every city for the most recent year -----
all_cities = sorted(composite["city"].unique())

results = []
for city in all_cities:
    latest_year = composite[composite["city"] == city]["year"].max()
    profile = get_risk_profile(city, latest_year)
    if "error" not in profile:
        results.append(profile)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("composite_score", ascending=False)

print("=" * 90)
print("  CLIMATE RISK REPORT — ALL CITIES (LATEST YEAR)")
print("=" * 90)
print(results_df[["city", "year", "composite_score", "composite_category", "bank_recommendation"]].to_string(index=False))